In [4]:
import logging

from pulao.logging import init_logging

from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData
from pulao.trend import Trend,TrendManager

import polars as pl

from pulao.logging import logger

init_logging(level=logging.DEBUG)
open("./logs/pulao.log", "w").close()  # 每次运行前清空日志
logger.debug("开始运行")

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.slice(0,200)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    o = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=o,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(
        symbol=bar.symbol,
        exchange=bar.exchange.value,
        interval=bar.interval.value,
        datetime=datetime,
        turnover=money,
        open_price=o,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
    )

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)
# 注入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

logger.debug("运行结束")


{"event": "开始运行", "level": "debug", "logger": "__main__", "time": "2025-12-01 01:47:25"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-12-01 01:47:25"}
{"swing_list": null, "event": "波段数量不足3个，跳过", "level": "debug", "logger": "__main__", "time": "2025-12-01 01:47:25"}
{"swing_list": null, "event": "波段数量不足3个，跳过", "level": "debug", "logger": "__main__", "time": "2025-12-01 01:47:25"}
{"cbar_count": 2, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-12-01 01:47:25"}
{"swing_list": null, "event": "波段数量不足3个，跳过", "level": "debug", "logger": "__main__", "time": "2025-12-01 01:47:25"}
{"fractal": "Fractal(left=CBar(id=121064580142071808, sbar_start_id=121064580129488896, sbar_end_id=121064580142071808, high_price=942.2349853515625, low_price=935.9240112304688, created_at=datetime.datetime(2025, 12, 1, 1, 47, 25, 177000), fractal_type=0), middle=CBar(id=121064580158849024, sbar_start_id=121064580158849024

In [5]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pulao.constant import *

#
# region . Plotly - Cbar
#
df_cbar = cbar_manager.df_cbar.to_pandas()
df_cbar["datetime"] = pd.date_range("2025-01-01", periods=df_cbar.shape[0], freq="h")
df_cbar["open_price"] = df_cbar["high_price"]
df_cbar["close_price"] = df_cbar["low_price"]
if "index" in df_cbar.columns:
    del df_cbar["index"]

df_cbar.insert(0, "index", range(len(df_cbar)))

# region 构建波段数据
df_swing = swing_manager.df_swing.to_pandas()
df_swing["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing["end_datetime"] = pd.Series(dtype="datetime64[ns]")
if "index" in df_swing.columns:
    del df_swing["index"]
df_swing.insert(0, "index", range(len(df_swing)))

for i, row in enumerate(df_swing.itertuples()):
    df = df_cbar[
        (df_cbar["id"] >= row.cbar_start_id) & (df_cbar["id"] <= row.cbar_end_id)
    ]
    start_datetime = df.iloc[0]["datetime"]
    end_datetime = df.iloc[-1]["datetime"]
    df_swing.at[i, "start_datetime"] = start_datetime
    df_swing.at[i, "end_datetime"] = end_datetime

xs_swing = []
ys_swing = []
texts_swing = []
text_positions_swing = []
for i, row in enumerate(df_swing.itertuples()):
    xs_swing += [row.start_datetime, row.end_datetime, None]
    print("df_swing - "+str(i),row)
    start_index = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["index"].iloc[0]
    end_index = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["index"].iloc[0]
    if row.direction == Direction.UP:
        start_price = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["low_price"].iloc[0]
        end_price = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["high_price"].iloc[0]
        text_positions_swing += ["bottom center", "top center", "middle center"]
    else:
        start_price = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["high_price"].iloc[
            0
        ]
        end_price = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["low_price"].iloc[0]
        text_positions_swing += ["top center", "bottom center", "middle center"]
    ys_swing += [start_price, end_price, None]
    texts_swing += [start_index, end_index, None]
# endregion

# region 构建趋势数据
df_trend = trend_manager.df_trend.to_pandas()
df_trend["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_trend["end_datetime"] = pd.Series(dtype="datetime64[ns]")
df_trend.insert(0, "index", range(len(df_trend)))

for i, row in enumerate(df_trend.itertuples()):
    df = df_swing[
        (df_swing["id"] >= row.swing_start_id) & (df_swing["id"] <= row.swing_end_id)
    ]
    start_datetime = df.iloc[0]["start_datetime"]
    end_datetime = df.iloc[-1]["end_datetime"]
    df_trend.at[i, "start_datetime"] = start_datetime
    df_trend.at[i, "end_datetime"] = end_datetime

xs_trend = []
ys_trend = []
texts_trend = []
text_positions_trend = []
for i, row in enumerate(df_trend.itertuples()):
    xs_trend += [row.start_datetime, row.end_datetime, None]
    print("df_trend - "+ str(i), row)
    start_index = df_swing[(df_swing["id"] == row.swing_start_id)]["index"].iloc[0]
    end_index = df_swing[(df_swing["id"] == row.swing_end_id)]["index"].iloc[0]
    if row.direction == Direction.UP:
        start_price = df_swing[(df_swing["id"] == row.swing_start_id)]["low_price"].iloc[0]
        end_price = df_swing[(df_swing["id"] == row.swing_end_id)]["high_price"].iloc[0]
        text_positions_trend += ["bottom center", "top center", "middle center"]
    else:
        start_price = df_swing[(df_swing["id"] == row.swing_start_id)]["high_price"].iloc[
            0
        ]
        end_price = df_swing[(df_swing["id"] == row.swing_end_id)]["low_price"].iloc[0]
        text_positions_trend += ["top center", "bottom center", "middle center"]
    ys_trend += [start_price, end_price, None]
    texts_trend += [start_index, end_index, None]
#endregion

# region fig
fig = make_subplots(rows=1, cols=1)
fig.add_trace(
    go.Candlestick(
        x=df_cbar["datetime"],
        open=df_cbar["open_price"],
        high=df_cbar["high_price"],
        low=df_cbar["low_price"],
        close=df_cbar["close_price"],
        name="K线",
    ),
    row=1,
    col=1,
)

df_fractal_bottom = df_cbar[(df_cbar["fractal_type"] == FractalType.BOTTOM)]

fig.add_trace(
    go.Scatter(
        x=df_fractal_bottom["datetime"],
        y=df_fractal_bottom["low_price"] * 0.995,  # 放在K线最低价下方一点，不遮挡蜡烛
        mode="markers+text",
        name="swing point low",
        marker=dict(
            symbol="triangle-up",
            size=14,
            color="#1E90FF",
        ),
        text=df_fractal_bottom["index"],  # ← 添加序号
        textposition="bottom center",  # ← 文字位置
        textfont=dict(size=10, color="white"),
        hovertemplate="<b>波段低点</b><br>"
        + "时间: %{x}<br>"
        + "价格: %{y:.2f}<br>"
        + "<extra></extra>",
    ),
    row=1,
    col=1,
)


df_fractal_top = df_cbar[(df_cbar["fractal_type"] == FractalType.TOP)]

fig.add_trace(
    go.Scatter(
        x=df_fractal_top["datetime"],
        y=df_fractal_top["high_price"] * 1.005,  # 放在K线最高价上方一点
        mode="markers+text",
        name="swing point high",
        marker=dict(
            symbol="triangle-down",
            size=14,
            color="#FF4500",
        ),
        text=df_fractal_top["index"],  # ← 添加序号
        textposition="top center",  # ← 文字位置
        textfont=dict(size=10, color="white"),
        hovertemplate="<b>波段高点</b><br>"
        + "时间: %{x}<br>"
        + "价格: %{y:.2f}<br>"
        + "<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=xs_swing,
        y=ys_swing,
        name="swing segments",
        mode="lines+text",  # lines+text 支持显示文字
        line=dict(width=2, color="orange"),
        text=texts_swing,
        textposition=text_positions_swing,  # 文字位置
    )
)

fig.add_trace(
    go.Scatter(
        x=xs_trend,
        y=ys_trend,
        name="trend segments",
        mode="lines+text",  # lines+text 支持显示文字
        line=dict(width=2, color="red"),
        text=texts_trend,
        textposition=text_positions_trend,  # 文字位置
    )
)

title = "Swing Chart"
fig.update_layout(
    title=title,
    height=900,
    hovermode="x unified",  # X轴悬停联动虚线
)

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=False,
)

fig.show()
# endregion

# endregion

df_swing - 0 Pandas(Index=0, index=0, id=121064580779606016, cbar_start_id=121064580158849024, cbar_end_id=121064580754440192, sbar_start_id=121064580158849024, sbar_end_id=121064580720885760, high_price=988.5280151367188, low_price=930.2979736328125, direction=1, is_completed=True, created_at=Timestamp('2025-12-01 01:47:25.329000'), start_datetime=Timestamp('2025-01-01 01:00:00'), end_datetime=Timestamp('2025-01-02 00:00:00'))
df_swing - 1 Pandas(Index=1, index=1, id=121064581572329472, cbar_start_id=121064580754440192, cbar_end_id=121064581542969344, sbar_start_id=121064580720885760, sbar_end_id=121064581521997824, high_price=988.5280151367188, low_price=906.1430053710938, direction=-1, is_completed=True, created_at=Timestamp('2025-12-01 01:47:25.518000'), start_datetime=Timestamp('2025-01-02 00:00:00'), end_datetime=Timestamp('2025-01-03 04:00:00'))
df_swing - 2 Pandas(Index=2, index=2, id=121064582105006080, cbar_start_id=121064581542969344, cbar_end_id=121064582037897216, sbar_sta

In [ ]:
from pulao.trend import Trend
from pulao.swing import Swing
from typing import List


class _TrendSFSeq:
    def __init__(self, sfs: List[Swing]=None, trend: Trend=None):
        self.sfs:List[Swing] = sfs # 特征序列
        self.trend:Trend = trend # 趋势
    def __bool__(self):
        return True if self.trend and self.sfs else False
    def clear(self):
        self.sfs = []
        self.trend = None
ts = _TrendSFSeq(sfs=[Swing()], trend=Trend())
# ts.clear()
if ts :
    print("ok")
else:
    print("not ok")

In [ ]:
from typing import List

# 测试swing 构建算法
from IPython.display import display
from pulao.logging import logger, init_logging
import logging
from time import sleep
from pulao.swing import Swing, SwingManager
from pulao.trend import Trend, TrendManager
from pulao.bar import CBar, CBarManager, SBarManager
from pulao.constant import *

init_logging(level=logging.DEBUG)
with open("./logs/pulao.log", "w"):
    pass  # 每次运行前清空日志

logger.debug("开始运行")


def format_df_trend():
    import polars as pl

    if "index" in swing_manager.df_swing.columns:
        swing_manager.df_swing = swing_manager.df_swing.drop("index")
    swing_manager.df_swing = swing_manager.df_swing.with_row_index("index")

    df_swing_tmp = swing_manager.df_swing.select(["id", "index"])
    df1 = trend_manager.df_trend.join(
        df_swing_tmp, left_on="swing_start_id", right_on="id", how="left"
    )
    df2 = trend_manager.df_trend.join(
        df_swing_tmp, left_on="swing_end_id", right_on="id", how="left"
    )
    df1 = df1.rename({"index": "swing_start_index"})
    df2 = df2.rename({"index": "swing_end_index"})
    df2 = df2.select(["id", "swing_end_index"])
    df1 = df1.join(df2, on="id", how="left")
    df = df1.select(
        [
            "id",
            "swing_start_index",
            "swing_end_index",
            "swing_start_id",
            "swing_end_id",
            "high_price",
            "low_price",
            "direction",
            "is_completed",
            "created_at",
        ]
    )

    return df


cbar_manager = CBarManager(sbar_manager=SBarManager())
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)
df_cbar = cbar_manager.read_parquet()
df_swing = swing_manager.read_parquet()
swing_manager.df_swing = swing_manager.df_swing.slice(0, 0)  # 清空原数据
for i in range(df_swing.height - 1):
    swing = Swing(**df_swing.row(i, named=True))
    swing_manager._append_swing(swing)
    swing_manager.notify(EventType.SWING_CHANGED)


logger.debug("运行结束")
sleep(0.2)


display("df_trend")
df = format_df_trend()
display(df)

display("df_swing")
swing_manager.df_swing

In [ ]:
swing_manager.df_swing = swing_manager.df_swing.drop("index")
swing_manager.next_swing(120701055628476416)

In [ ]:
# 可视化日志
import pandas as pd

df = pd.read_json("./logs/pulao.log", lines=True)
df = df.drop(columns=["logger", "level", "time"])
priority_cols = ["event"]
other_cols = [c for c in df.columns if c not in priority_cols]
df = df[priority_cols + other_cols]
df["info"] = df[other_cols].apply(lambda row: row.dropna().tolist(), axis=1)
df = df.drop(columns=other_cols)
df["info"] = df["info"].apply(lambda x: "" if x == [] else x)
df

In [ ]:
cbar_manager.read_parquet()